# WDC Schema.org Class-Specific Subset — Domain-Targeted Extraction

This notebook:
1. Fetches the **lookup CSV** for every WDC 2024-12 class-specific subset
2. Identifies which **chunk files** contain data from our target domains
3. Downloads **only those chunks**, filters for matching quads, and deletes the download
4. Saves the filtered quads per class to `output/<ClassName>.nq`

This avoids downloading the full ~1.4 TB corpus — we only fetch the chunks we actually need.

## 1. Configuration

In [14]:
import os
import io
import gzip
import json
import time
import random
from pathlib import Path
from collections import defaultdict

import pandas as pd
import requests

# ── Core domains (manually curated, always targeted) ────────────
CORE_DOMAINS = [
    # Knowledge Base
    "en.wikipedia.org",
    # Dedicated fact-checkers
    "politifact.com",
    "snopes.com",
    "factcheck.org",
    "fullfact.org",
    "africacheck.org",
    # News services
    "reuters.com",
    "apnews.com",
    # International broadcasters
    "bbc.com",
    "bbc.co.uk",
    "aljazeera.com",
    "france24.com",
    "dw.com",
    # Major newspapers
    "theguardian.com",
    "nytimes.com",
    "thetimes.com",
    "lemonde.fr",
    "cnn.com",
]

# ── Merge with merged_domains.csv → final_domains.csv ───────────
MERGED_DOMAINS_CSV = Path("../SourceDomainsExtraction/merged_domains.csv")
FINAL_DOMAINS_CSV  = Path("../SourceDomainsExtraction/final_domains.csv")

existing_df = pd.read_csv(MERGED_DOMAINS_CSV)
existing_domains = set(existing_df["domain"].str.strip().str.lower())

new_rows = [
    {"domain": d, "source": "core"}
    for d in CORE_DOMAINS
    if d.lower() not in existing_domains
]
if "source" not in existing_df.columns:
    existing_df["source"] = "merged"

final_df = pd.concat(
    [existing_df, pd.DataFrame(new_rows)],
    ignore_index=True
).drop_duplicates(subset="domain")
final_df.to_csv(FINAL_DOMAINS_CSV, index=False)

print(f"merged_domains.csv : {len(existing_df):>4} domains")
print(f"Core additions     : {len(new_rows):>4} new domains added")
print(f"final_domains.csv  : {len(final_df):>4} domains total")
if new_rows:
    print(f"  Added: {[r['domain'] for r in new_rows]}")

# ── Build TARGET_DOMAINS from final_domains.csv ──────────────────
TARGET_DOMAINS = final_df["domain"].tolist()

# ── Build TARGET_PLDS (pay-level domains for WDC lookup matching) ─
TARGET_PLDS = set()
for d in TARGET_DOMAINS:
    TARGET_PLDS.add(d.lower())
    parts = d.split(".")
    if len(parts) > 2:
        if parts[-2] in ("co", "com", "org", "net", "edu", "gov", "ac"):
            pass  # bbc.co.uk is already the PLD
        else:
            TARGET_PLDS.add(".".join(parts[-2:]).lower())  # en.wikipedia.org → wikipedia.org

# ── Build DOMAIN_PATTERNS for line-level N-Quad filtering ────────
DOMAIN_PATTERNS = []
for d in TARGET_DOMAINS:
    DOMAIN_PATTERNS.append(f"://{d}/")
    DOMAIN_PATTERNS.append(f"://{d}>")
    DOMAIN_PATTERNS.append(f".{d}/")
    DOMAIN_PATTERNS.append(f".{d}>")

print(f"\nTarget domains : {len(TARGET_DOMAINS)}")
print(f"Target PLDs    : {len(TARGET_PLDS)}")


merged_domains.csv :  499 domains
Core additions     :    8 new domains added
final_domains.csv  :  507 domains total
  Added: ['en.wikipedia.org', 'factcheck.org', 'africacheck.org', 'apnews.com', 'bbc.co.uk', 'aljazeera.com', 'france24.com', 'thetimes.com']

Target domains : 507
Target PLDs    : 509


In [15]:
# ── All 50 WDC class-specific subsets ───────────────────────────
BASE_URL = "https://data.dws.informatik.uni-mannheim.de/structureddata/2024-12/quads/classspecific"

SCHEMA_CLASSES = [
    "AdministrativeArea", "Airport", "Answer", "Book", "City",
    "ClaimReview", "CollegeOrUniversity", "Continent", "Country",
    "CreativeWork", "Dataset", "EducationalOrganization", "Event",
    "FAQPage", "GeoCoordinates", "GovernmentOrganization", "Hospital",
    "Hotel", "JobPosting", "LakeBodyOfWater", "LandmarksOrHistoricalBuildings",
    "Language", "Library", "LocalBusiness", "Mountain", "Movie",
    "Museum", "MusicAlbum", "MusicRecording", "Organization", "Painting",
    "Park", "Person", "Place", "Product", "QAPage", "Question",
    "RadioStation", "Recipe", "Restaurant", "RiverBodyOfWater", "School",
    "SearchAction", "ShoppingCenter", "SkiResort", "SportsEvent",
    "SportsTeam", "StadiumOrArena", "TVEpisode", "TelevisionStation",
]

# ── Directories ─────────────────────────────────────────────────
OUTPUT_DIR = Path("output")
TEMP_DIR = Path("wdc_temp")
LOOKUP_DIR = Path("lookups")
CHECKPOINT_FILE = Path("checkpoint_lookup.json")

OUTPUT_DIR.mkdir(exist_ok=True)
TEMP_DIR.mkdir(exist_ok=True)
LOOKUP_DIR.mkdir(exist_ok=True)

# ── Session ─────────────────────────────────────────────────────
session = requests.Session()
session.headers.update({
    "User-Agent": "WDC-Research-Filter/2.0 (academic research; "
                  "contact: your-email@example.com)"
})

# ── Rate-limit settings ─────────────────────────────────────────
PAUSE_BETWEEN_DOWNLOADS = 10   # seconds between chunk downloads
MAX_RETRIES = 6

print(f"Classes: {len(SCHEMA_CLASSES)}")
print(f"Output: {OUTPUT_DIR}/")

Classes: 50
Output: output/


## 2. Helper functions

In [16]:
def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, "r") as f:
            return json.load(f)
    return {"completed_classes": [], "completed_chunks": {}, "stats": {}}


def save_checkpoint(state):
    tmp = str(CHECKPOINT_FILE) + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f, indent=2)
    os.replace(tmp, CHECKPOINT_FILE)


def download_with_retry(url, dest_path=None, max_retries=MAX_RETRIES):
    """
    Download a URL with retry + 429 backoff.
    If dest_path is given, saves to disk. Otherwise returns response content.
    """
    for attempt in range(max_retries):
        try:
            resp = session.get(url, timeout=300, stream=(dest_path is not None))

            if resp.status_code == 429:
                retry_after = resp.headers.get("Retry-After")
                if retry_after:
                    wait = int(retry_after) + random.uniform(5, 15)
                else:
                    wait = (120 * (2 ** attempt)) + random.uniform(0, 30)
                print(f"    ⏳ 429, waiting {wait:.0f}s (attempt {attempt+1})...")
                time.sleep(wait)
                continue

            resp.raise_for_status()

            if dest_path:
                with open(dest_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
                return True
            else:
                return resp.content

        except (requests.RequestException, IOError) as e:
            wait = (60 * (2 ** attempt)) + random.uniform(0, 15)
            print(f"    ⚠ Error (attempt {attempt+1}): {e}")
            if attempt < max_retries - 1:
                print(f"      Retrying in {wait:.0f}s...")
                time.sleep(wait)

    return None



def line_matches_domain(line):
    line = line.rstrip()
    # Graph URI is the last <...> before the trailing " ."
    last_close = line.rfind('>')
    last_open = line.rfind('<', 0, last_close)
    if last_open == -1:
        return False
    graph_uri = line[last_open:last_close + 1]
    for pattern in DOMAIN_PATTERNS:
        if pattern in graph_uri:
            return True
    return False


print("Helpers loaded.")

Helpers loaded.


## 3. Fetch all lookup tables & identify relevant chunks

In [ ]:
def fetch_lookup(cls_name):
    """
    Download the lookup CSV for a class and return a DataFrame.
    The CSV maps PLDs to chunk file numbers.
    Expected columns: pld (or domain), file (or chunk/part number).
    """
    local_path = LOOKUP_DIR / f"{cls_name}_lookup.csv"

    # Use cached version if available
    if local_path.exists() and local_path.stat().st_size > 0:
        try:
            return pd.read_csv(local_path)
        except Exception:
            pass

    url = f"{BASE_URL}/{cls_name}/{cls_name}_lookup.csv"
    content = download_with_retry(url)

    if content is None:
        print(f"  ✗ Could not download lookup for {cls_name}")
        return None

    # Save locally for caching
    with open(local_path, "wb") as f:
        f.write(content)

    try:
        return pd.read_csv(io.BytesIO(content))
    except Exception as e:
        print(f"  ✗ Could not parse lookup for {cls_name}: {e}")
        return None


# Fetch all lookups and find which chunks contain our domains
class_chunk_map = {}  # {class_name: {chunk_file: [matching_plds]}}
classes_with_no_match = []

print("Fetching lookup tables for all classes...\n")

for i, cls in enumerate(SCHEMA_CLASSES):
    print(f"[{i+1:>2}/{len(SCHEMA_CLASSES)}] {cls}...", end=" ")

    df = fetch_lookup(cls)

    if df is None:
        print("SKIP (download failed)")
        continue

    # Detect column names (varies slightly across releases)
    cols_lower = {c.lower().strip(): c for c in df.columns}

    # Find the PLD/domain column
    pld_col = None
    for candidate in ["pld", "domain", "host", "pay-level-domain", "pay_level_domain"]:
        if candidate in cols_lower:
            pld_col = cols_lower[candidate]
            break
    if pld_col is None:
        # Fall back to first column
        pld_col = df.columns[0]

    # Find the file/chunk column
    file_col = None
    for candidate in ["file", "file_lookup", "chunk", "part", "filename", "file_name"]:
        if candidate in cols_lower:
            file_col = cols_lower[candidate]
            break
    if file_col is None:
        # Fall back to second column
        file_col = df.columns[1]

    # Match our target PLDs (exact match only)
    matching = defaultdict(list)
    for _, row in df.iterrows():
        pld = str(row[pld_col]).strip().lower()
        chunk = row[file_col]

        if pld in TARGET_PLDS:
            # Normalize chunk reference to filename
            chunk_str = str(chunk).strip()
            if not chunk_str.startswith("part_"):
                chunk_str = f"part_{chunk_str}"
            if not chunk_str.endswith(".gz"):
                chunk_str = f"{chunk_str}.gz"
            matching[chunk_str].append(pld)

    if matching:
        class_chunk_map[cls] = dict(matching)
        total_plds = sum(len(v) for v in matching.values())
        print(f"✓ {total_plds} PLDs across {len(matching)} chunks")
    else:
        classes_with_no_match.append(cls)
        print(f"— no target domains found")

    # Small pause between lookup downloads
    if i < len(SCHEMA_CLASSES) - 1:
        time.sleep(1)

print(f"\n{'═' * 60}")
print(f"Classes with matches: {len(class_chunk_map)}")
print(f"Classes without matches: {len(classes_with_no_match)}")
total_chunks = sum(len(v) for v in class_chunk_map.values())
print(f"Total chunk files to download: {total_chunks}")

Fetching lookup tables for all classes...

[ 1/50] AdministrativeArea... ✓ 6 PLDs across 5 chunks
[ 2/50] Airport... — no target domains found
[ 3/50] Answer... ✓ 75 PLDs across 51 chunks
[ 4/50] Book... ✓ 18 PLDs across 11 chunks
[ 5/50] City... ✓ 4 PLDs across 3 chunks
[ 6/50] ClaimReview... ✓ 15 PLDs across 1 chunks
[ 7/50] CollegeOrUniversity... ✓ 3 PLDs across 3 chunks
[ 8/50] Continent... — no target domains found
[ 9/50] Country... ✓ 27 PLDs across 23 chunks
[10/50] CreativeWork... 

since there are a lot of chunks to download:
- we can filter top 10 chunks per class (that include more domains than others)
- make sure both fact-checking sites and news are represented, specially the top ones in each (in fac-check we can use the count col in SourceDomainsExtraction/Fact_Check/top_15_org_domains.csv and in news we can use stories_per_week in SourceDomainsExtraction/News/national_sources_cpw_gt_900_20260424.csv)
- we can add a sense of parallelism by downloading another chunk while the previous one is being processed (unziped and filtered) so we make it faster without parallel requests to wdc.

In [ ]:
# ── Priority-based chunk selection ───────────────────────────
import csv
from collections import defaultdict

TOP_CHUNKS_PER_CLASS = 10
CORE_MAX_CLASSES     = 5   # each core PLD extracted from at most this many classes

def _to_pld(domain):
    """Strip one subdomain level to get pay-level domain (mirrors TARGET_PLDS logic)."""
    parts = domain.lower().split(".")
    if len(parts) > 2 and parts[-2] not in ("co", "com", "org", "net", "edu", "gov", "ac"):
        return ".".join(parts[-2:])
    return domain.lower()

# PLDs derived from the core curated list — these must always be represented
CORE_PLDS = {_to_pld(d) for d in CORE_DOMAINS}

# Schema classes ranked by fact-checking relevance (lower index = higher priority)
FACT_CHECK_CLASS_ORDER = [
    # Tier 1: directly fact-checking / Q&A
    "ClaimReview", "FAQPage", "QAPage", "Question", "Answer",
    # Tier 2: news & information
    "Event", "CreativeWork", "SearchAction", "Dataset", "Book",
    # Tier 3: entities (supporting context)
    "Organization", "Person", "Place", "Country", "City",
    "AdministrativeArea", "CollegeOrUniversity", "GovernmentOrganization",
    # Tier 4: lower relevance
    "Product", "LocalBusiness", "Restaurant", "Recipe", "Hotel", "JobPosting",
    "Movie", "MusicRecording", "MusicAlbum", "SportsEvent", "SportsTeam",
    "GeoCoordinates", "Language", "StadiumOrArena", "TVEpisode", "RadioStation",
    "EducationalOrganization", "Hospital", "LandmarksOrHistoricalBuildings",
    "ShoppingCenter", "Park",
]

def _class_rank(cls):
    try: return FACT_CHECK_CLASS_ORDER.index(cls)
    except ValueError: return len(FACT_CHECK_CLASS_ORDER)

# ── Fact-check and news scores (for filling non-core slots) ──────
FACT_CHECK_CSV = Path("../SourceDomainsExtraction/Fact_Check/top_15_org_domains.csv")
fact_pld_score = {}
with open(FACT_CHECK_CSV) as _f:
    for _row in csv.DictReader(_f):
        _pld = _to_pld(_row["org_domain"])
        fact_pld_score[_pld] = fact_pld_score.get(_pld, 0) + int(_row["count"])

NEWS_CSV = Path("../SourceDomainsExtraction/News/national_sources_cpw_gt_900_20260424.csv")
news_pld_score = {}
with open(NEWS_CSV) as _f:
    for _row in csv.DictReader(_f):
        _name = _row.get("name", "").strip()
        if not _name: continue
        _pld = _to_pld(_name)
        try: _spw = float(_row.get("stories_per_week") or 0)
        except ValueError: _spw = 0
        news_pld_score[_pld] = max(news_pld_score.get(_pld, 0), _spw)

_max_fact = max(fact_pld_score.values(), default=1)
_max_news = max(news_pld_score.values(), default=1)
NEWS_WEIGHT = _max_fact / _max_news

def _chunk_score(plds):
    score  = sum(fact_pld_score.get(p, 0) for p in plds)
    score += sum(news_pld_score.get(p, 0) * NEWS_WEIGHT for p in plds)
    return score + len(plds) * 0.01

# ── Step 1: per-class selection — core chunks guaranteed ──────────
filtered_chunk_map = {}
print("Step 1 — Per-class selection (core PLDs always included):")
print(f"  {'Class':<35} {'Before':>7}  {'After':>6}  Core PLDs")
print("  " + "─" * 72)

for _cls, _chunks in class_chunk_map.items():
    _items = list(_chunks.items())
    if len(_items) <= TOP_CHUNKS_PER_CLASS:
        filtered_chunk_map[_cls] = _chunks
        _core_in = sorted(CORE_PLDS & {p for ps in _chunks.values() for p in ps})
        print(f"  {_cls:<35} {len(_items):>7}  {len(_items):>6}  {', '.join(_core_in) or '—'}")
        continue

    _scored      = sorted(_items, key=lambda kv: _chunk_score(kv[1]), reverse=True)
    _core_chunks = {cf: plds for cf, plds in _scored if CORE_PLDS & set(plds)}
    _other       = [(cf, plds) for cf, plds in _scored if cf not in _core_chunks]
    _selected    = dict(_core_chunks)
    for cf, plds in _other[:max(TOP_CHUNKS_PER_CLASS - len(_core_chunks), 0)]:
        _selected[cf] = plds

    filtered_chunk_map[_cls] = _selected
    _core_found = sorted(CORE_PLDS & {p for ps in _core_chunks.values() for p in ps})
    print(f"  {_cls:<35} {len(_items):>7}  {len(_selected):>6}  {', '.join(_core_found) or '—'}")

# ── Step 2: cross-class pruning — keep each core PLD in its ──────
# top-K most fact-checking-relevant classes only
print(f"\nStep 2 — Cross-class pruning (each core PLD kept in top {CORE_MAX_CLASSES} classes):")
print(f"  {'Core PLD':<25} {'Kept in':<45} Dropped from")
print("  " + "─" * 90)

_core_pld_classes = defaultdict(list)
for _cls, _chunks in filtered_chunk_map.items():
    for _plds in _chunks.values():
        for _p in _plds:
            if _p in CORE_PLDS and _cls not in _core_pld_classes[_p]:
                _core_pld_classes[_p].append(_cls)

for _pld, _classes in sorted(_core_pld_classes.items(), key=lambda x: -len(x[1])):
    _sorted_cls = sorted(_classes, key=_class_rank)
    _keep = set(_sorted_cls[:CORE_MAX_CLASSES])
    _drop = set(_sorted_cls[CORE_MAX_CLASSES:])
    for _cls in _drop:
        filtered_chunk_map[_cls] = {
            cf: plds for cf, plds in filtered_chunk_map[_cls].items()
            if _pld not in plds
        }
    _kept_str = ', '.join(sorted(_keep, key=_class_rank))
    _drop_str = ', '.join(sorted(_drop, key=_class_rank)) if _drop else '—'
    print(f"  {_pld:<25} {_kept_str:<45} {_drop_str}")

# Remove classes that became empty after pruning
filtered_chunk_map = {cls: chunks for cls, chunks in filtered_chunk_map.items() if chunks}

_orig  = sum(len(v) for v in class_chunk_map.values())
_final = sum(len(v) for v in filtered_chunk_map.values())
print(f"\n{'─'*50}")
print(f"Original chunks : {_orig}")
print(f"After filtering : {_final}  ({(_orig-_final)/_orig*100:.1f}% reduction)")


In [ ]:
# ── Summary of what we'll download (after priority filtering) ──
print(f"{'Class':<35} {'Chunks':>7}  {'PLDs (fact-check★  news◆  other)'}")
print("─" * 80)
for cls, chunks in sorted(filtered_chunk_map.items()):
    plds = {p for plist in chunks.values() for p in plist}
    labelled = []
    for p in sorted(plds):
        if p in fact_pld_score:
            labelled.append(f"{p}★")
        elif p in news_pld_score:
            labelled.append(f"{p}◆")
        else:
            labelled.append(p)
    print(f"{cls:<35} {len(chunks):>7}  {', '.join(labelled)}")

print(f"\nTotal chunks to download: {sum(len(v) for v in filtered_chunk_map.values())}")


Class                                Chunks  PLDs (fact-check★  news◆  other)
────────────────────────────────────────────────────────────────────────────────
AdministrativeArea                        5  freepressjournal.in◆, ig.com.br◆, indianexpress.com◆, lefigaro.fr◆, pravda.ru◆, rbc.ua◆
Answer                                   10  aif.ru◆, amarujala.com◆, ambito.com◆, cbsnews.com◆, cbssports.com◆, donya-e-eqtesad.com◆, elcomercio.pe◆, elcorreo.com◆, elperiodico.com◆, eluniversal.com.mx◆, excite.co.jp◆, express.co.uk◆, hindustantimes.com◆, independent.co.uk◆, independent.ie◆, indiatimes.com◆, inquirer.com◆, mk.ru◆, onet.pl◆, prtimes.jp◆, ria.ru◆, stern.de◆, stheadline.com◆, zeit.de◆
Book                                     10  aawsat.com◆, alkhaleej.ae◆, athensvoice.gr◆, cna.com.tw◆, elcorreo.com◆, globo.com◆, idnes.cz◆, lanacion.com.ar◆, latercera.com◆, lavanguardia.com◆, laverdad.es◆, lefigaro.fr◆, novilist.hr◆, okaz.com.sa◆, segre.com◆, tienphong.vn◆, udn.com◆
City               

## 4. Download, filter, and save

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def download_chunk_file(cls_name, chunk_file, pre_delay=0):
    """Download a chunk after an optional rate-limit delay. Returns local_path or None."""
    if pre_delay > 0:
        time.sleep(pre_delay)
    url = f"{BASE_URL}/{cls_name}/{chunk_file}"
    local_path = TEMP_DIR / chunk_file
    success = download_with_retry(url, dest_path=local_path)
    return local_path if success else None


def filter_chunk_file(local_path, output_path):
    """Decompress, filter for target-domain lines, append matches, delete file.
    Returns (matches, total_lines)."""
    matches = 0
    total_lines = 0
    matched_lines = []
    try:
        with gzip.open(local_path, "rt", encoding="utf-8", errors="replace") as f:
            for line in f:
                total_lines += 1
                if line_matches_domain(line):
                    matched_lines.append(line)
                    matches += 1
    except Exception as e:
        print(f"    ⚠ Error reading {local_path}: {e}")
        return -1, total_lines
    finally:
        if local_path and local_path.exists():
            local_path.unlink()
    if matched_lines:
        with open(output_path, "a", encoding="utf-8") as out:
            out.writelines(matched_lines)
    return matches, total_lines


def process_chunk(cls_name, chunk_file, output_path):
    """Combined download + filter (used when pipelining is off)."""
    local_path = download_chunk_file(cls_name, chunk_file)
    if local_path is None:
        return -1, 0
    return filter_chunk_file(local_path, output_path)


print("Processing functions ready.")


Processing functions ready.


In [ ]:
# ── Save chunk plans to disk ─────────────────────────────────
CHUNK_PLAN_FILE          = Path("chunk_plan.json")
CHUNK_PLAN_FILTERED_FILE = Path("chunk_plan_filtered.json")

with open(CHUNK_PLAN_FILE, "w") as f:
    json.dump(class_chunk_map, f, indent=2)

with open(CHUNK_PLAN_FILTERED_FILE, "w") as f:
    json.dump(filtered_chunk_map, f, indent=2)

_orig = sum(len(v) for v in class_chunk_map.values())
_new  = sum(len(v) for v in filtered_chunk_map.values())
print(f"Full plan   → {CHUNK_PLAN_FILE}          ({len(class_chunk_map)} classes, {_orig} chunks)")
print(f"Filtered    → {CHUNK_PLAN_FILTERED_FILE}  ({len(filtered_chunk_map)} classes, {_new} chunks)")


Full plan   → chunk_plan.json          (38 classes, 1264 chunks)
Filtered    → chunk_plan_filtered.json  (38 classes, 231 chunks)


In [ ]:
# ── Main processing loop (pipelined: download N+1 while filtering N) ──
state = load_checkpoint()
completed_classes = set(state.get("completed_classes", []))
completed_chunks  = state.get("completed_chunks", {})
stats             = state.get("stats", {})

classes_to_process = [
    cls for cls in filtered_chunk_map if cls not in completed_classes
]

print(f"Classes to process: {len(classes_to_process)} "
      f"(already done: {len(completed_classes)})")
print(f"{'═' * 65}\n")

for cls_idx, cls_name in enumerate(classes_to_process):
    chunks      = filtered_chunk_map[cls_name]
    output_path = OUTPUT_DIR / f"{cls_name}.nq"

    pending = [
        (cf, plds) for cf, plds in chunks.items()
        if f"{cls_name}/{cf}" not in completed_chunks
    ]

    print(f"\n── {cls_name} ({cls_idx+1}/{len(classes_to_process)}) "
          f"— {len(pending)} pending / {len(chunks)} total chunks ──")

    if not pending:
        print("  All chunks already processed, skipping.")
        state["completed_classes"].append(cls_name)
        save_checkpoint(state)
        continue

    cls_matches = cls_lines = 0
    cls_failed  = []
    done_before = len(chunks) - len(pending)  # already-completed count

    with ThreadPoolExecutor(max_workers=1) as dl_executor:
        # Kick off download of first pending chunk immediately
        next_future = dl_executor.submit(download_chunk_file, cls_name, pending[0][0], 0)

        for chunk_idx, (chunk_file, plds) in enumerate(pending):
            chunk_key   = f"{cls_name}/{chunk_file}"
            display_idx = done_before + chunk_idx + 1

            # Wait for this chunk's download
            t0         = time.time()
            local_path = next_future.result()
            dl_time    = time.time() - t0

            # Schedule next download (with rate-limit delay) *before* filtering,
            # so download overlaps with the filter work below
            if chunk_idx + 1 < len(pending):
                _delay      = PAUSE_BETWEEN_DOWNLOADS + random.uniform(0, 5)
                next_future = dl_executor.submit(
                    download_chunk_file, cls_name, pending[chunk_idx + 1][0], _delay
                )

            # Filter current chunk (runs while next chunk download+delay is in progress)
            t1 = time.time()
            if local_path is not None:
                matches, total_lines = filter_chunk_file(local_path, output_path)
            else:
                matches, total_lines = -1, 0
            filter_time = time.time() - t1

            if matches >= 0:
                cls_matches += matches
                cls_lines   += total_lines
                completed_chunks[chunk_key] = True
                status = f"✓ {matches:,} matches" if matches else "  0 matches"
                print(f"  [{display_idx}/{len(chunks)}] {chunk_file}  {status}  "
                      f"({total_lines:,} lines, dl={dl_time:.0f}s filter={filter_time:.0f}s)  "
                      f"PLDs: {', '.join(plds)}")
            else:
                cls_failed.append(chunk_file)
                print(f"  [{display_idx}/{len(chunks)}] {chunk_file}  ✗ FAILED  "
                      f"PLDs: {', '.join(plds)}")

            state["completed_chunks"] = completed_chunks
            save_checkpoint(state)

    if not cls_failed:
        state["completed_classes"].append(cls_name)

    out_size = output_path.stat().st_size / 1024 if output_path.exists() else 0
    stats[cls_name] = {
        "matches":          cls_matches,
        "lines_scanned":    cls_lines,
        "chunks_processed": len(pending) - len(cls_failed),
        "chunks_failed":    len(cls_failed),
        "output_kb":        round(out_size, 1),
    }
    state["stats"] = stats
    save_checkpoint(state)

    print(f"  Summary: {cls_matches:,} matches from {cls_lines:,} lines "
          f"→ {output_path} ({out_size:.1f} KB)")
    if cls_failed:
        print(f"  ⚠ Failed chunks: {cls_failed}")

    if cls_idx < len(classes_to_process) - 1:
        _pause = PAUSE_BETWEEN_DOWNLOADS * 2 + random.uniform(0, 10)
        print(f"  ⏸  Pausing {_pause:.0f}s before next class...")
        time.sleep(_pause)

print(f"\n{'═' * 65}")
print("All classes processed!")


Classes to process: 38 (already done: 0)
═════════════════════════════════════════════════════════════════


── AdministrativeArea (1/38) — 5 pending / 5 total chunks ──


## 5. Summary

In [ ]:
# ── Final summary ──────────────────────────────────────────────
state = load_checkpoint()
stats = state.get("stats", {})

if stats:
    summary_rows = []
    for cls_name, s in sorted(stats.items()):
        if s["matches"] > 0:
            summary_rows.append({
                "Class": cls_name,
                "Matches": s["matches"],
                "Lines Scanned": s["lines_scanned"],
                "Chunks": s["chunks_processed"],
                "Failed": s["chunks_failed"],
                "Output (KB)": s["output_kb"],
            })

    df_summary = pd.DataFrame(summary_rows)
    if not df_summary.empty:
        print("\nClasses with matching quads:\n")
        print(df_summary.to_string(index=False))
        print(f"\nTotal matches: {df_summary['Matches'].sum():,}")
        print(f"Total output:  {df_summary['Output (KB)'].sum():.1f} KB")
    else:
        print("No matches found in any class.")

    # Classes with 0 matches
    zero_classes = [c for c, s in stats.items() if s["matches"] == 0]
    if zero_classes:
        print(f"\nClasses with 0 matches: {', '.join(sorted(zero_classes))}")
else:
    print("No stats recorded yet — run the processing cell first.")


Classes with matching quads:

       Class  Matches  Lines Scanned  Chunks  Failed  Output (KB)
      Answer      551       25068272       2       0        140.3
 ClaimReview    75924        3919703       1       0      15965.4
CreativeWork    74967       37631469       3       0      15762.6
       Event      623       12529590       1       0        179.8
     FAQPage      551       25096515       2       0        140.3
  JobPosting      718       12536476       1       0        211.3
Organization  2097769      113346990       6       0     414935.7
      Person  2389601      100454336       8       0     625080.5
       Place    12770       41581019       3       0       2183.9
     Product     5423       12533384       1       0       1246.8
    Question      551       25659063       2       0        140.3
SearchAction   299195       62849976       5       0      75949.5
 SportsEvent    11719       12535401       1       0       1844.1
  SportsTeam    11497       16211819       1 

In [ ]:
# ── List output files ──────────────────────────────────────────
print("\nOutput files:\n")
for f in sorted(OUTPUT_DIR.glob("*.nq")):
    size = f.stat().st_size
    if size > 0:
        # Count lines
        with open(f, "r") as fh:
            n_lines = sum(1 for _ in fh)
        if size > 1024 * 1024:
            print(f"  {f.name:<40} {size / (1024*1024):.1f} MB  ({n_lines:,} quads)")
        else:
            print(f"  {f.name:<40} {size / 1024:.1f} KB  ({n_lines:,} quads)")


Output files:

  Answer.nq                                140.3 KB  (551 quads)
  ClaimReview.nq                           15.6 MB  (75,924 quads)
  CreativeWork.nq                          15.4 MB  (74,967 quads)
  Event.nq                                 179.8 KB  (623 quads)
  FAQPage.nq                               140.3 KB  (551 quads)
  JobPosting.nq                            211.3 KB  (718 quads)
  Organization.nq                          405.2 MB  (2,109,799 quads)
  Person.nq                                610.4 MB  (2,389,601 quads)
  Place.nq                                 2.1 MB  (12,770 quads)
  Product.nq                               1.2 MB  (5,423 quads)
  Question.nq                              140.3 KB  (551 quads)
  SearchAction.nq                          74.2 MB  (299,195 quads)
  SportsEvent.nq                           1.8 MB  (11,719 quads)
  SportsTeam.nq                            1.8 MB  (11,497 quads)


## 6. Quick peek at results

In [ ]:
# Show first 5 lines from each non-empty output file
for f in sorted(OUTPUT_DIR.glob("*.nq")):
    if f.stat().st_size > 0:
        print(f"\n{'─' * 60}")
        print(f"  {f.name}")
        print(f"{'─' * 60}")
        with open(f, "r") as fh:
            for i, line in enumerate(fh):
                if i >= 5:
                    print(f"  ... (truncated)")
                    break
                # Truncate long lines for display
                display = line.strip()
                if len(display) > 120:
                    display = display[:120] + "..."
                print(f"  {display}")


────────────────────────────────────────────────────────────
  Answer.nq
────────────────────────────────────────────────────────────
  <https://www.mg4law.com/#Organization> <http://schema.org/areaServed> "https://en.wikipedia.org/wiki/Conyers,_Georgia" <...
  <https://www.mg4law.com/#Organization> <http://schema.org/areaServed> "https://en.wikipedia.org/wiki/Covington,_Georgia"...
  <https://www.mg4law.com/#Organization> <http://schema.org/areaServed> "https://en.wikipedia.org/wiki/Atlanta" <https://w...
  <https://www.mgmotor.co.in/vehicles/comet-ev-electric-car-in-india> <http://schema.org/sameAs> <https://en.wikipedia.org...
  <https://www.mgmotor.co.in/vehicles/mghector-plus-6-seater> <http://schema.org/sameAs> <https://en.wikipedia.org/wiki/MG...
  ... (truncated)

────────────────────────────────────────────────────────────
  ClaimReview.nq
────────────────────────────────────────────────────────────
  _:n8988ab806129453cbf92733aa7a2e832xb0 <http://schema.org/claimReviewed> "&

In [ ]:
# ── Merge + deduplicate all .nq files, collect URI stats ────────
from pathlib import Path
from urllib.parse import urlparse
from collections import Counter

OUTPUT_DIR = Path("output")
MERGED_FILE = OUTPUT_DIR / "merged.nq"

def _extract_graph_uri(line):
    line = line.rstrip()
    last_close = line.rfind('>')
    if last_close == -1:
        return None
    last_open = line.rfind('<', 0, last_close)
    if last_open == -1:
        return None
    return line[last_open + 1:last_close]

def _parse_host(uri):
    try:
        host = urlparse(uri).netloc.lower()
        parts = host.split(".")
        if len(parts) >= 3 and parts[-2] in ("co", "com", "org", "net", "edu", "gov", "ac"):
            reg = ".".join(parts[-3:])
        else:
            reg = ".".join(parts[-2:]) if len(parts) >= 2 else host
        return host, reg
    except Exception:
        return None, None

seen_hashes = set()
uri_records = []          # (class_name, full_uri, host, registered_domain)
per_class_counts = {}
total_dups = 0

print(f"{'Class':<20} {'total':>10}  {'unique':>10}  {'dups':>8}")
print("─" * 55)

with open(MERGED_FILE, "w", encoding="utf-8") as out:
    for nq_file in sorted(OUTPUT_DIR.glob("*.nq")):
        if nq_file.name == "merged.nq":
            continue
        cls_name = nq_file.stem
        cls_total = cls_unique = 0

        with open(nq_file, encoding="utf-8") as f:
            for line in f:
                cls_total += 1
                h = hash(line)
                if h not in seen_hashes:
                    seen_hashes.add(h)
                    out.write(line)
                    cls_unique += 1
                    uri = _extract_graph_uri(line)
                    if uri:
                        host, reg = _parse_host(uri)
                        if host:
                            uri_records.append((cls_name, uri, host, reg))
                else:
                    total_dups += 1

        per_class_counts[cls_name] = {"total": cls_total, "unique": cls_unique}
        print(f"{cls_name:<20} {cls_total:>10,}  {cls_unique:>10,}  {total_dups:>8,}")

total_read = sum(v["total"]  for v in per_class_counts.values())
total_uniq = sum(v["unique"] for v in per_class_counts.values())
print(f"\n{'='*55}")
print(f"Total read:    {total_read:>10,}")
print(f"Duplicates:    {total_dups:>10,}  ({total_dups/total_read*100:.2f}%)")
print(f"Unique quads:  {total_uniq:>10,}")
print(f"Merged → {MERGED_FILE}  ({MERGED_FILE.stat().st_size / 1e6:.1f} MB)")


Class                     total      unique      dups
───────────────────────────────────────────────────────
Answer                      551         551         0
ClaimReview              75,924      75,898        26
CreativeWork             74,967           0    74,993
Event                       623         598    75,018
FAQPage                     551           0    75,569
JobPosting                  718         718    75,569
Organization          2,109,799   2,067,365   118,003
Person                2,389,601   1,892,252   615,352
Place                    12,770      11,497   616,625
Product                   5,423           0   622,048
Question                    551           0   622,599
SearchAction            299,195     104,129   817,665
SportsEvent              11,719         222   829,162
SportsTeam               11,497           0   840,659

Total read:     4,993,889
Duplicates:       840,659  (16.83%)
Unique quads:   4,153,230
Merged → output/merged.nq  (982.2 MB)


In [ ]:
# ── Build stats DataFrame ────────────────────────────────────────
import pandas as pd

df_uris = pd.DataFrame(uri_records, columns=["class", "uri", "host", "registered_domain"])

print(f"Unique quads with parseable source URI: {len(df_uris):,}")
print(f"Unique hosts (subdomains):              {df_uris['host'].nunique():,}")
print(f"Unique registered domains:              {df_uris['registered_domain'].nunique():,}")
print()

# Quads per registered domain
domain_totals = df_uris["registered_domain"].value_counts()
print("Quads per registered domain:")
print(f"  {'domain':<35} {'quads':>9}  {'%':>6}")
print(f"  {'-'*52}")
for domain, count in domain_totals.items():
    pct = count / len(df_uris) * 100
    print(f"  {domain:<35} {count:>9,}  {pct:>5.1f}%")


Unique quads with parseable source URI: 4,153,230
Unique hosts (subdomains):              28
Unique registered domains:              10

Quads per registered domain:
  domain                                  quads       %
  ----------------------------------------------------
  cnn.com                             1,915,302   46.1%
  wikipedia.org                       1,240,187   29.9%
  dw.com                                826,324   19.9%
  factcheck.org                          90,145    2.2%
  fullfact.org                           38,215    0.9%
  politifact.com                         20,875    0.5%
  aljazeera.com                          12,410    0.3%
  france24.com                            7,842    0.2%
  thetimes.com                            1,862    0.0%
  theguardian.com                            68    0.0%


In [ ]:
# ── Subdomain distribution ───────────────────────────────────────
from collections import Counter

print(f"{'='*65}")
print(f"Subdomains per registered domain  (merged, deduplicated quads)")
print(f"{'='*65}\n")

domain_hosts = df_uris.groupby("registered_domain")["host"].apply(list)

for domain, hosts in sorted(domain_hosts.items(), key=lambda x: -len(x)):
    host_counts = Counter(hosts)
    print(f"{domain}  ({len(hosts):,} quads total)")
    print(f"  {'subdomain':<45} {'count':>8}  {'%':>6}")
    print(f"  {'-'*62}")
    for host, count in host_counts.most_common():
        pct = count / len(hosts) * 100
        print(f"  {host:<45} {count:>8,}  {pct:>5.1f}%")
    print()


Subdomains per registered domain  (merged, deduplicated quads)

aljazeera.com  (12,410 quads total)
  subdomain                                        count       %
  --------------------------------------------------------------
  www.aljazeera.com                               12,243   98.7%
  interactive.aljazeera.com                          161    1.3%
  mediaview.aljazeera.com                              6    0.0%

cnn.com  (1,915,302 quads total)
  subdomain                                        count       %
  --------------------------------------------------------------
  cnnespanol.cnn.com                            1,772,239   92.5%
  amp.cnn.com                                     88,967    4.6%
  arabic.cnn.com                                  33,116    1.7%
  cnne-stage.cnn.com                              11,241    0.6%
  commercial.cnn.com                               8,670    0.5%
  cnne-test.cnn.com                                  568    0.0%
  sponsorcontent.cnn

In [ ]:
# ── All schema.org types × domain (from rdf:type triples in merged) ──
# how many type declarations of each class per domain

RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
type_records = []

import re

def normalize_schema_type(uri):
    return re.sub(r"^https?://(?:www\.)?schema\.org/", "", uri)


with open(MERGED_FILE, encoding="utf-8") as f:
    for line in f:
        if RDF_TYPE not in line:
            continue
        parts = line.split()
        if len(parts) < 4 or parts[1] != RDF_TYPE:
            continue
        uri = _extract_graph_uri(line)
        if not uri:
            continue
        _, reg = _parse_host(uri)
        if not reg:
            continue
        schema_type = normalize_schema_type(parts[2].strip("<>"))
        type_records.append((reg, schema_type))

df_types = pd.DataFrame(type_records, columns=["registered_domain", "schema_type"])

print(f"Type declarations: {len(df_types):,}")
print(f"Distinct schema.org types: {df_types['schema_type'].nunique()}\n")

pivot_all = (
    df_types.groupby(["registered_domain", "schema_type"])
    .size()
    .unstack(fill_value=0)
)
pivot_all.insert(0, "TOTAL", pivot_all.sum(axis=1))
pivot_all = pivot_all.sort_values("TOTAL", ascending=False)
print(pivot_all.to_string())

# Top 20 most-annotated source pages (unchanged)
print(f"\n{'='*70}")
print("Top 20 source pages by quad count")
print(f"{'='*70}")
print(f"  {'quads':>6}  url")
print(f"  {'-'*66}")
top_pages = df_uris.groupby("uri").size().sort_values(ascending=False).head(20)
for uri, count in top_pages.items():
    display = uri if len(uri) <= 80 else uri[:77] + "..."
    print(f"  {count:>6,}  {display}")


Type declarations: 816,664
Distinct schema.org types: 43

schema_type         TOTAL  AggregateOffer  Answer  Article  BlogPosting  BreadcrumbList  BroadcastEvent  Claim  ClaimReview  CollectionPage  CreativeWork  EntryPoint  Event  FAQPage  ImageGallery  ImageObject  JobPosting  ListItem  LiveBlogPosting  MediaObject  NewsArticle  NewsMediaOrganization  Offer  Organization  Person  Place  PostalAddress  Product  ProfilePage  PropertyValue  PropertyValueSpecification  Question  Rating  ReadAction  SearchAction  SearchResultsPage  SiteNavigationElement  SpeakableSpecification  SportsEvent  SportsTeam  VideoObject  WebPage  WebPageElement  WebSite
registered_domain                                                                                                                                                                                                                                                                                                                                          

In [ ]:
# ── Unique annotated pages per domain ────────────────────────────
ann = (
    df_uris.groupby("registered_domain")
    .agg(
        quads=("uri", "count"),
        pages=("uri", "nunique"),
    )
    .sort_values("quads", ascending=False)
)
ann["quads_per_page"] = (ann["quads"] / ann["pages"]).round(1)

print(f"{'domain':<35} {'pages':>8}  {'quads':>10}  {'quads/page':>11}")
print("─" * 68)
for domain, row in ann.iterrows():
    print(f"{domain:<35} {row['pages']:>8,}  {row['quads']:>10,}  {row['quads_per_page']:>11.1f}")


domain                                 pages       quads   quads/page
────────────────────────────────────────────────────────────────────
cnn.com                             18,825.0  1,915,302.0        101.7
wikipedia.org                       66,739.0  1,240,187.0         18.6
dw.com                              41,245.0   826,324.0         20.0
factcheck.org                        1,602.0    90,145.0         56.3
fullfact.org                         1,158.0    38,215.0         33.0
politifact.com                         725.0    20,875.0         28.8
aljazeera.com                          334.0    12,410.0         37.2
france24.com                           237.0     7,842.0         33.1
thetimes.com                           118.0     1,862.0         15.8
theguardian.com                          2.0        68.0         34.0


In [ ]:
# ── Actual schema.org types present in each class file ───────────
import re
from collections import Counter
from pathlib import Path

OUTPUT_DIR = Path("output")
RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"

def normalize_schema_type(uri):
    uri = re.sub(r"^https?://(?:www\.)?schema\.org/", "", uri)
    uri = uri.replace("http://www.w3.org/1999/02/22-rdf-syntax-ns#", "rdf:")
    return uri

print(f"{'='*65}")
print("Schema.org types found inside each class file")
print(f"{'='*65}")

for nq_file in sorted(OUTPUT_DIR.glob("*.nq")):
    if nq_file.name == "merged.nq":
        continue
    type_counts = Counter()
    with open(nq_file, encoding="utf-8") as f:
        for line in f:
            if RDF_TYPE not in line:
                continue
            parts = line.split()
            if len(parts) >= 4 and parts[1] == RDF_TYPE:
                label = normalize_schema_type(parts[2].strip("<>"))
                type_counts[label] += 1

    print(f"\n{nq_file.stem}  ({sum(type_counts.values()):,} type triples)")
    print(f"  {'type':<55} {'count':>8}")
    print(f"  {'-'*65}")
    for t, count in type_counts.most_common(20):
        print(f"  {t:<55} {count:>8,}")


Schema.org types found inside each class file

Answer  (112 type triples)
  type                                                       count
  -----------------------------------------------------------------
  ImageObject                                                   23
  Product                                                       21
  AggregateOffer                                                21
  Offer                                                         19
  Organization                                                  11
  Question                                                       4
  Answer                                                         4
  NewsMediaOrganization                                          2
  FAQPage                                                        1
  BreadcrumbList                                                 1
  ListItem                                                       1
  NewsArticle                                         

In [ ]:
# ── Actual schema.org types in merged (deduplicated) quads ───────
RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
type_counts = Counter()

with open(MERGED_FILE, encoding="utf-8") as f:
    for line in f:
        if RDF_TYPE not in line:
            continue
        parts = line.split()
        if len(parts) >= 4 and parts[1] == RDF_TYPE:
            label = normalize_schema_type(parts[2].strip("<>"))
            type_counts[label] += 1

print(f"Type triples in merged file: {sum(type_counts.values()):,}\n")
print(f"  {'type':<55} {'count':>8}")
print(f"  {'-'*65}")
for t, count in type_counts.most_common(30):
    print(f"  {t:<55} {count:>8,}")


Type triples in merged file: 816,664

  type                                                       count
  -----------------------------------------------------------------
  Organization                                             263,128
  ImageObject                                              176,129
  Article                                                   69,588
  Person                                                    61,283
  WebPage                                                   59,857
  NewsArticle                                               58,059
  NewsMediaOrganization                                     30,890
  VideoObject                                               16,267
  WebPageElement                                            15,085
  ListItem                                                   9,429
  MediaObject                                                9,123
  SiteNavigationElement                                      7,020
  BlogPosting          